<a href="https://colab.research.google.com/github/Shineii86/MoeStickerBot/blob/main/notebooks/MoeStickerBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://capsule-render.vercel.app/api?type=waving&height=300&color=gradient&text=𝗠𝗼𝗲%20𝗦𝘁𝗶𝗰𝗸𝗲𝗿%20𝗕𝗼𝘁&fontAlignY=30&fontSize=100&desc=𝖢𝗈𝗅𝖺𝖻%20𝖤𝖽𝗂𝗍𝗂𝗈𝗇%20—%20𝖲𝖾𝗅𝖿‑𝖧𝗈𝗌𝗍%20𝖸𝗈𝗎𝗋%20𝖳𝖾𝗅𝖾𝗀𝗋𝖺𝗆%20𝖲𝗍𝗂𝖼𝗄𝖾𝗋%20𝖡𝗈𝗍&descSize=30" alt="Moe Sticker Bot">
  <p><b>Import LINE/Kakao · Create · Manage — all in one notebook</b></p>
</div>

---

## 🎯 Features

| Feature | Description |
|---------|-------------|
| 📥 **Import** | Import LINE & KakaoTalk sticker packs (including animated!) into Telegram |
| 🎨 **Create** | Create your own sticker sets from any image or video |
| 🛠️ **Manage** | Edit, reorder, add/remove stickers from your sets |
| 💾 **Download** | Download any Telegram sticker or GIF |
| 🔍 **Search** | Search previously imported sticker packs |

---

## 🤖 Bot Commands

| Command | What It Does |
|---------|-------------|
| `/start` | Welcome message & instructions |
| `/import` | Import a LINE or Kakao sticker pack |
| `/create` | Create a new sticker set from your images/videos |
| `/manage` | Edit your existing sticker sets |
| `/download` | Download Telegram stickers or GIFs |
| `/search` | Search imported sticker packs by keyword |
| `/help` | Show all available commands |

**💡 Tip:** Just send a LINE/Kakao link directly — the bot auto-detects it!

**Example links:**
```
https://store.line.me/stickershop/product/7673/ja
https://e.kakao.com/t/pretty-all-friends
https://emoticon.kakao.com/items/lV6K2fWmU7CpXlHcP9-ysQJx9rg=?referer=share_link
```

---

In [ ]:
#@title 🚀 Moe Sticker Bot — 配置与启动
# ╔══════════════════════════════════════════════════════════════════════╗
# ║          🚀 MOE STICKER BOT — Configuration & Launch               ║
# ║          ALL-IN-ONE CELL  ·  No UI boxes  ·  Full ANSI             ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ==================== 配置（在此编辑） ====================
ENABLE_DB       = True            # TiDB Cloud shared database (sticker data persists for everyone)
ENABLE_WEBAPP   = False           # Set True to expose WebApp via ngrok
WEBAPP_PORT     = 8080
NGROK_AUTHTOKEN = ""              # Required if ENABLE_WEBAPP=True
DATA_DIR        = "moe_sticker_bot_data"
LOG_LEVEL       = "info"          # debug, info, warn, error
HTTP_PROXY      = ""              # Optional http://proxy:port
AUTO_RESTART    = True            # Auto-restart bot on crash
MAX_RESTARTS    = 5               # Max restart attempts before giving up
KEEP_ALIVE      = True            # Print heartbeat every 10 min (fights Colab timeout)
# ===================================================================

# ========== 加密的 BOT TOKEN（请勿编辑） ==========
import base64 as _b64
_ENC_TOKEN = "dVhTY01aV15VREkDLjIAZQJsBAEtFjBREVwzLTY6FVlICGFTBB4dHwcTICAoJw=="
_a=bytes().fromhex("4d6f65537469");_b=bytes().fromhex("636b657273426f7432303236");_k=(_a+_b).decode()
BOT_TOKEN="".join(chr(ord(c)^ord(_k[i%len(_k)]))for i,c in enumerate(_b64.b64decode(_ENC_TOKEN).decode("latin-1")));del _a,_b,_k
# ========================================================

# ┌──────────────────────────────────────────────────────────────────┐
# │  📦 导入 & 依赖                                                  │
# │  标准库、第三方模块和进度条                                       │
# └──────────────────────────────────────────────────────────────────┘
import sys, time, subprocess, os, urllib.request, json, threading, signal, atexit
from itertools import cycle
from tqdm.notebook import tqdm

# ┌──────────────────────────────────────────────────────────────────┐
# │  🎨 ANSI 颜色类 & 输出辅助                                       │
# │  终端样式、加载动画和格式化消息                                   │
# └──────────────────────────────────────────────────────────────────┘
class C:
    R = '\033[0m'; B = '\033[1m'; D = '\033[2m'
    BLK = '\033[30m'; RD = '\033[31m'; GN = '\033[32m'; YL = '\033[33m'
    BL = '\033[34m'; MG = '\033[35m'; CY = '\033[36m'; WH = '\033[37m'
    BRD = '\033[91m'; BGN = '\033[92m'; BYL = '\033[93m'; BBL = '\033[94m'
    BMG = '\033[95m'; BCY = '\033[96m'; BWH = '\033[97m'
    BGRD = '\033[41m'; BGGN = '\033[42m'; BGYL = '\033[43m'; BGBL = '\033[44m'

def success(m): print(f"{C.B}{C.BGGN}{C.BLK} ✓ {m} {C.R}")
def error(m):   print(f"{C.B}{C.BGRD}{C.WH} ✗ {m} {C.R}")
def info(m):    print(f"{C.B}{C.BGBL}{C.WH} ℹ {m} {C.R}")
def warn(m):    print(f"{C.B}{C.BGYL}{C.BLK} ⚠ {m} {C.R}")
def header(t):  print(f"\n{C.B}{C.BCY}{'═'*50}\n  {t}\n{'═'*50}{C.R}\n")

def spinner(msg, dur=2):
    frames = cycle(['⠋','⠙','⠹','⠸','⠼','⠴','⠦','⠧','⠇','⠏'])
    end = time.time() + dur
    while time.time() < end:
        sys.stdout.write(f'\r{C.BCY}{next(frames)} {msg}{C.R}  ')
        sys.stdout.flush()
        time.sleep(0.1)
    sys.stdout.write(f'\r{C.BGN}✔{C.R} {msg}   \n')

# ┌──────────────────────────────────────────────────────────────────┐
# │  🖥️  启动横幅                                                    │
# │  显示欢迎界面和机器人状态信息                                     │
# └──────────────────────────────────────────────────────────────────┘
print(f"{C.B}{C.BCY}")
print("  ╔══════════════════════════════════════════════════════╗")
print("  ║        🚀  Moe Sticker Bot — Colab 版              ║")
print("  ║   导入 LINE/Kakao · 创建 · 管理 · 下载             ║")
print("  ╠══════════════════════════════════════════════════════╣")
print(f"  ║  {C.R}{C.BGN}{C.BLK} ✓ Pre-configured {C.R}{C.BCY}  Bot token · DB · Deps        ║")
print(f"  ║  {C.R}{C.BGRD}{C.WH} ⚠ WARNING        {C.R}{C.BCY}  Don't edit — just run!      ║")
print(f"  ║  {C.R}{C.D}  @MoeStickersBot  →  t.me/MoeStickersBot         {C.R}{C.BCY} ║")
print("  ╚══════════════════════════════════════════════════════╝")
print(f"{C.R}")
print(f"  {C.GN}✓{C.R} Bot Token  : 已加密并锁定")
print(f"  {C.GN}✓{C.R} 数据库    : TiDB Cloud（共享）")
print(f"  {C.GN}✓{C.R} 保活机制  : 每 10 分钟心跳")
print(f"  {C.GN}✓{C.R} 自动重启  : 崩溃最多重试 {MAX_RESTARTS} 次")
print(f"\n{C.B}{C.BGN}✨ 准备就绪！开始设置...{C.R}\n")

# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧  阶段 1 — 环境设置 & 构建机器人                              ║
# ║  安装依赖 · 连接数据库 · 下载 Go · 编译二进制                     ║
# ╚══════════════════════════════════════════════════════════════════╝

# ┌── 📥 安装系统依赖 ───────────────────────────────────────────────┐
header("安装系统依赖")
spinner("Updating packages", 1)
!apt-get update -qq 2>/dev/null
!apt-get install -y -qq imagemagick libarchive-tools ffmpeg curl gifsicle python3 exiv2 2>/dev/null
success("核心包安装完成")

# ┌── 🗄️  连接 TiDB Cloud 数据库 ────────────────────────────────────┐
if ENABLE_DB:
    header("连接 TiDB Cloud 数据库")
    !which mysql >/dev/null 2>&1 || apt-get install -y -qq mysql-client 2>/dev/null
    _tidb_host = "gateway01.ap-southeast-1.prod.aws.tidbcloud.com"
    _tidb_port = "4000"
    _tidb_user = "3yvemDZyfJVpUSg.root"
    _tidb_pass = "JnamvZsoCC7cOitX"
    _tidb_db   = "MoeStickersBot_db"
    _test = subprocess.run(
        ["mysql", "-h", _tidb_host, "-P", _tidb_port,
         "-u", _tidb_user, f"-p{_tidb_pass}", "--ssl-mode=REQUIRED",
         "-e", f"USE `{_tidb_db}`;"],
        capture_output=True, text=True)
    if _test.returncode == 0:
        success(f"TiDB Cloud 已连接 — {C.B}{_tidb_db}{C.R}（共享）")
    else:
        warn(f"数据库连接: {_test.stderr.strip()[:100]}")
else:
    info("数据库已禁用")

# ┌── 🐹 下载 & 设置 Go 语言 ────────────────────────────────────────┐
GO_VERSION = "go1.22.4"
GO_URL = f"https://go.dev/dl/{GO_VERSION}.linux-amd64.tar.gz"
if not os.path.exists("/usr/local/go/bin/go") or GO_VERSION not in subprocess.getoutput("go version"):
    print(f"{C.CY}⬇ Downloading {GO_VERSION}...{C.R}")
    with tqdm(unit='B', unit_scale=True, desc=f"{C.BCY}Go{C.R}") as t:
        urllib.request.urlretrieve(GO_URL, "go.tar.gz", reporthook=lambda b,bs,total: t.update(b*bs-t.n))
    !rm -rf /usr/local/go && tar -C /usr/local -xzf go.tar.gz
else:
    info(f"{GO_VERSION} already installed — skipping download")
os.environ['PATH'] += ":/usr/local/go/bin"
os.environ['GOPATH'] = "/root/go"
os.environ['GO111MODULE'] = "on"
os.environ['GOFLAGS'] = "-mod=mod"
!mkdir -p $GOPATH
success(f"Go {subprocess.getoutput('go version').split()[2]} 已就绪")

# ┌── 🐍 安装 Python 辅助脚本 ───────────────────────────────────────┐
header("安装 Python 辅助脚本")
helpers = [("msb_emoji.py","Emoji"), ("msb_kakao_decrypt.py","Kakao"), ("msb_rlottie.py","Lottie")]
for f,desc in helpers:
    !wget -q https://raw.githubusercontent.com/Shineii86/MoeStickersBot/master/tools/{f} -O /usr/local/bin/{f}
    !chmod +x /usr/local/bin/{f}
    print(f"  {C.GN}✓{C.R} {desc}")
success("辅助脚本安装完成")

# ┌── 🔨 编译 MoeStickersBot 二进制文件 ─────────────────────────────┐
header("编译 MoeStickersBot")
!rm -rf MoeStickersBot
!git clone --depth 1 https://github.com/Shineii86/MoeStickersBot.git 2>&1 | grep -v "Cloning"
%cd MoeStickersBot
spinner("Downloading Go modules", 2)
!go mod download
spinner("Compiling binary", 3)
!go build -ldflags="-s -w" -o MoeStickersBot cmd/MoeStickersBot/main.go
if os.path.exists("MoeStickersBot"):
    sz = os.path.getsize("MoeStickersBot")/1024/1024
    success(f"编译完成 — 二进制文件: {sz:.1f} MB")
else:
    error("编译失败 — 请检查上方 git clone / go build 输出")
    sys.exit(1)

# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  阶段 2 — 配置 & 准备启动                                   ║
# ║  验证 Token · ngrok 隧道 · 构建命令行 · 日志着色                 ║
# ╚══════════════════════════════════════════════════════════════════╝

if not BOT_TOKEN:
    error("BOT_TOKEN 未设置！请编辑此单元格顶部的变量后重新运行。")
    sys.exit(1)

import re
if not re.match(r'^\d+:[A-Za-z0-9_-]{35,}$', BOT_TOKEN):
    warn("BOT_TOKEN 格式看起来不对 — 应为  123456:ABC…（数字:35+字符）")

header("配置信息")
print(f"  {C.GN}✓{C.R} BOT_TOKEN    = {BOT_TOKEN[:8]}...{BOT_TOKEN[-4:]}")
print(f"  {C.GN}✓{C.R} LOG_LEVEL    = {LOG_LEVEL}")
print(f"  {C.GN}✓{C.R} DATA_DIR     = {DATA_DIR}")
print(f"  {C.GN}✓{C.R} AUTO_RESTART = {AUTO_RESTART}  (max {MAX_RESTARTS}x)")
print(f"  {C.GN}✓{C.R} KEEP_ALIVE   = {KEEP_ALIVE}")

if ENABLE_WEBAPP and not NGROK_AUTHTOKEN:
    warn("WebApp 已启用但未提供 ngrok token — 已禁用")
    ENABLE_WEBAPP = False

# ┌── 🌐 设置 ngrok 隧道（如启用 WebApp）────────────────────────────┐
WEBAPP_URL = ""
ngrok_proc = None
if ENABLE_WEBAPP:
    header("设置 ngrok 隧道")
    if not os.path.exists("./ngrok"):
        !wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz && tar -xzf ngrok*.tgz && chmod +x ngrok
    !./ngrok config add-authtoken {NGROK_AUTHTOKEN}
    !pkill -f ngrok || true
    ngrok_proc = subprocess.Popen(["./ngrok", "http", str(WEBAPP_PORT), "--log", "stdout"],
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    spinner("Starting ngrok", 3)
    for _ in range(15):
        try:
            import requests
            r = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=2)
            if r.status_code == 200:
                tuns = r.json()['tunnels']
                if tuns:
                    WEBAPP_URL = tuns[0]['public_url']
                    success(f"ngrok URL: {WEBAPP_URL}")
                    break
        except: pass
        time.sleep(1)
    else:
        error("无法获取 ngrok URL — WebApp 已禁用")
        ENABLE_WEBAPP = False

# ┌── 🧩 构建机器人命令行参数 ───────────────────────────────────────┐
def build_cmd():
    cmd = ["./MoeStickersBot",
           f"--bot_token={BOT_TOKEN}",
           f"--log_level={LOG_LEVEL}",
           f"--data_dir={DATA_DIR}"]
    if ENABLE_DB:
        cmd.extend(["--db_addr=gateway01.ap-southeast-1.prod.aws.tidbcloud.com:4000", "--db_user=3yvemDZyfJVpUSg.root", "--db_pass=JnamvZsoCC7cOitX"])
    if ENABLE_WEBAPP and WEBAPP_URL:
        cmd += [f"--webapp_url={WEBAPP_URL}", f"--webapp_listen_addr=0.0.0.0:{WEBAPP_PORT}"]
    return cmd

if HTTP_PROXY:
    os.environ['HTTP_PROXY'] = HTTP_PROXY
    os.environ['HTTPS_PROXY'] = HTTP_PROXY

cmd_line = build_cmd()
print(f"{C.D}Command: {' '.join(cmd_line).replace(BOT_TOKEN, '[REDACTED]')}{C.R}")

# ┌── 🌈 日志着色（ANSI 格式化）─────────────────────────────────────┐
def colorize(line):
    line = line.replace('INFO',    f'{C.BGBL}{C.WH} INFO {C.R}')
    line = line.replace('WARNING', f'{C.BGYL}{C.BLK} WARN {C.R}')
    line = line.replace('ERROR',   f'{C.BGRD}{C.WH} ERROR {C.R}')
    line = line.replace('DEBUG',   f'{C.D} DEBUG {C.R}')
    line = line.replace('Bot OK',  f'{C.BGN}{C.B}Bot OK{C.R}')
    line = line.replace('one sticker commited',      f'{C.GN}✔ committed{C.R}')
    line = line.replace('Failed to add one sticker', f'{C.BRD}{C.B}✘ failed{C.R}')
    line = line.replace('Success',             f'{C.BGN}{C.B}Success{C.R}')
    line = line.replace('STICKER_VIDEO_LONG',  f'{C.BYL}{C.B}STICKER_VIDEO_LONG{C.R}')
    line = line.replace('safe mode',           f'{C.MG}safe mode{C.R}')
    line = line.replace('convertKakaoAnimated OK', f'{C.GN}convert OK{C.R}')
    return line

# ┌── 🧹 清理 & 关闭处理 ────────────────────────────────────────────┐
_process_ref = [None]

def cleanup():
    p = _process_ref[0]
    if p and p.poll() is None:
        p.terminate()
        try: p.wait(timeout=5)
        except subprocess.TimeoutExpired: p.kill()
    if ngrok_proc and ngrok_proc.poll() is None:
        ngrok_proc.terminate()
    print(f"\n{C.BGYL}正在关闭...{C.R}")
    success("机器人已停止。")

atexit.register(cleanup)

# ┌── 💓 保活心跳线程 ───────────────────────────────────────────────┐
_ka_stop = threading.Event()
def _keep_alive():
    beat = 0
    while not _ka_stop.wait(600):
        beat += 1
        print(f"{C.D}[保活心跳 #{beat} — {time.strftime('%H:%M:%S')}]{C.R}")
if KEEP_ALIVE:
    threading.Thread(target=_keep_alive, daemon=True).start()

# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀  阶段 3 — 启动机器人（带自动重启）                           ║
# ║  启动进程 · 流式日志 · 崩溃自动重启                               ║
# ╚══════════════════════════════════════════════════════════════════╝
header("启动机器人")

restart_count = 0
while True:
    process = subprocess.Popen(
        build_cmd(),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        cwd=os.getcwd(), bufsize=1, universal_newlines=True
    )
    _process_ref[0] = process

    spinner("Starting bot", 2)
    time.sleep(2)

    if process.poll() is not None:
        error("机器人立即退出 — 请检查 BOT_TOKEN 和构建输出")
        sys.exit(1)

    success(f"机器人正在运行 — PID {process.pid}"
            + (f"  [restart #{restart_count}]" if restart_count else ""))
    print(f"{C.B}{C.BGN}📱 在 Telegram 上向你的 bot 发送 /start！{C.R}")
    if WEBAPP_URL:
        print(f"{C.B}{C.BCY}🌐 WebApp: {WEBAPP_URL}{C.R}")
    print()

    header("实时日志 — 按 ■（停止）终止")
    print(f"  {C.D}日志在下方实时显示。{C.R}\n")
    try:
        for line in process.stdout:
            line = line.rstrip()
            if line:
                print(colorize(line))
    except KeyboardInterrupt:
        warn("用户中断")
        break

    rc = process.wait()
    if not AUTO_RESTART or restart_count >= MAX_RESTARTS:
        warn(f"机器人退出（代码 {rc}）。AUTO_RESTART={AUTO_RESTART}，重启次数 {restart_count}/{MAX_RESTARTS}。")
        break

    restart_count += 1
    warn(f"机器人退出（代码 {rc}）— 5 秒后重启…（第 {restart_count}/{MAX_RESTARTS} 次）")
    time.sleep(5)

_ka_stop.set()
cleanup()

---

## 🔧 Troubleshooting

| 问题 | 解决方案 |
|------|----------|
| **"Bot exited immediately"** | Check if your `BOT_TOKEN` is correct |
| **"Database not enabled"** | Normal! Bot works fine without database |
| **Bot stops after ~90 min** | Free Colab disconnects — `KEEP_ALIVE=True` helps; use Colab Pro for longer sessions |
| **Sticker import fails** | Try again — Telegram rate-limits sometimes. Bot auto-retries |
| **"STICKER_VIDEO_LONG"** | Bot handles this automatically via safe mode |
| **WebApp not working** | Set `ENABLE_WEBAPP = True` + valid `NGROK_AUTHTOKEN` |
| **"go build" fails** | Run cell again — Go module download may have timed out |
| **Bot keeps restarting** | Set `AUTO_RESTART = False` to debug; check logs above |

---

## 💡 Pro Tips

1. **Keep Colab Alive** — `KEEP_ALIVE = True` prints a heartbeat every 10 min; keep tab open and interact every 30-45 min
2. **Auto-Restart** — `AUTO_RESTART = True` + `MAX_RESTARTS = 5` recovers from crashes automatically
3. **Animated Kakao** — Use share links (KakaoTalk → Share → Copy Link) for animation support
4. **Mixed Sets** — Put animated + static stickers in the same Telegram sticker set
5. **Re-run to Update** — Run the cell again to pull latest bot code and rebuild
6. **Debug Mode** — Set `LOG_LEVEL = "debug"` for detailed logs
7. **Faster Builds** — Binary is now stripped (`-ldflags="-s -w"`) — smaller & faster to start

---

## ❓ FAQ

**Q: Is this free?** → Yes! Colab is free, bot is open-source.

**Q: 24/7 hosting?** → Free Colab disconnects after ~90 min. Use a VPS for 24/7.

**Q: Need WebApp?** → No, optional. Bot works fully without it.

**Q: Stickers saved forever?** → Yes! Once in Telegram, they stay even if bot goes offline.

**Q: What changed in this version?**
- Go upgraded to **1.22.4** (auto-skips re-download if already installed)
- **Auto-restart** on crash with configurable max attempts
- **Keep-alive** heartbeat thread to fight Colab timeouts
- BOT_TOKEN **format validation** before launch
- `HTTP_PROXY` now sets both `HTTP_PROXY` and `HTTPS_PROXY`
- Binary built with `-ldflags="-s -w"` — stripped & smaller
- ngrok tunnel wait extended to 15 retries with timeout
- `atexit` cleanup — bot stops cleanly even on unexpected exits

---

<div align="center">
  <img src="https://capsule-render.vercel.app/api?type=waving&color=gradient&customColorList=12,14,20,24,27&height=100&section=footer" width="100%">
  <p>Made with ❤️ for the sticker community</p>
</div>


---

## 🔐 仅限管理员：更新 Bot Token

> ⚠️ **注意！** 此部分仅限 **bot 管理员**使用。
> 
> **Regular users:** Skip this. Your bot is already configured.
> 
> **If you're not @Shineii86 — DO NOT touch this section.**

If you need to change the bot token, run the cell below with your new token.

In [ ]:
#@title 🔐 仅限管理员：加密 / 解密 Bot Token
#@markdown ---
#@markdown ### 🔑 Owner Verification
#@markdown Enter your secret owner key to unlock this tool.
OWNER_KEY = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### ⚙️ Action
ACTION = "encrypt"  #@param ["encrypt", "decrypt"]
#@markdown *`encrypt` = plain token → encrypted string · `decrypt` = encrypted string → plain token*

#@markdown ---
#@markdown ### 🔓 Encrypt — Paste your raw bot token here
#@markdown *Get one from [@BotFather](https://t.me/BotFather) → `/newbot`*
PLAIN_TOKEN = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### 🔒 Decrypt — Paste your encrypted token here
ENC_TOKEN = ""  #@param {type:"string"}

#@markdown ---

# ┌──────────────────────────────────────────────────────────────────┐
# │  📦 IMPORTS                                                      │
# │  Cryptographic hashing, encoding, and IPython display utils      │
# └──────────────────────────────────────────────────────────────────┘
import base64, hashlib, re, sys
from IPython.display import display, HTML

_OWNER_HASH = "e67795aea9e86b1b694e226456cb703bb48e6e5daa819bae4dae204c7c0cd571"

# ┌──────────────────────────────────────────────────────────────────┐
# │  🔐 ENCRYPTION / DECRYPTION CORE FUNCTIONS                       │
# │  XOR cipher with rotating key, token format validation           │
# └──────────────────────────────────────────────────────────────────┘

def _xor(text, k):
    return "".join(chr(ord(c) ^ ord(k[i % len(k)])) for i, c in enumerate(text))

def _cipher_key():
    _a = bytes().fromhex("4d6f65537469")
    _b = bytes().fromhex("636b657273426f7432303236")
    return (_a + _b).decode()

def encrypt_token(raw):
    k = _cipher_key()
    return base64.b64encode(_xor(raw, k).encode("latin-1")).decode()

def decrypt_token(enc):
    k = _cipher_key()
    return _xor(base64.b64decode(enc).decode("latin-1"), k)

# ┌──────────────────────────────────────────────────────────────────┐
# │  ▶️  MAIN EXECUTION — Verify Key & Process Token                 │
# │  Owner verification → encrypt or decrypt based on ACTION         │
# └──────────────────────────────────────────────────────────────────┘

print("═" * 50)
print("  🔐 仅限管理员 Token 工具")
print("═" * 50)

if not OWNER_KEY:
    print("❌ 请在上方输入你的 Owner Key 后重新运行。")
elif hashlib.sha256(OWNER_KEY.encode()).hexdigest() != _OWNER_HASH:
    print("🔒 访问被拒绝 — 密钥错误。")
else:
    print("  ✅ 密钥验证成功！")
    print()

    if ACTION == "encrypt":
        if not PLAIN_TOKEN:
            print("❌ PLAIN_TOKEN 为空。请在上方粘贴你的原始 bot token。")
        else:
            result = encrypt_token(PLAIN_TOKEN)
            print("═" * 50)
            print("  ✅ 加密后的 Token — 请复制下方内容")
            print("═" * 50)
            print(f"  {result}")
            print()
            print("📋 在主单元格中找到：")
            print('     _ENC_TOKEN = "..."'
)
            print(f'   替换为：\n     _ENC_TOKEN = "{result}"')
            print("═" * 50)

    elif ACTION == "decrypt":
        if not ENC_TOKEN:
            print("❌ ENC_TOKEN 为空。请在上方粘贴你的加密字符串。")
        else:
            try:
                result = decrypt_token(ENC_TOKEN)
                valid  = bool(re.match(r"^\d+:[A-Za-z0-9_-]{35,}$", result))
                status = "✅ Valid Telegram token format" if valid else "⚠️  Format looks unusual — double-check"
                print("═" * 50)
                print("  🔓 解密后的 Token")
                print("═" * 50)
                print(f"  {result}")
                print()
                print(f"  {status}")
                print("═" * 50)
            except Exception as e:
                print(f"❌ 解密失败： {e}")
